# Gold Layer Data Validation
This notebook runs data integrity tests on the `gold_hourly_clinical` table to ensure the Medallion Pipeline correctly transformed the event-driven data into a continuous ML-ready time series.

In [1]:
import duckdb
import pandas as pd

# Connect to the Lakehouse DB (read-only to avoid lock conflicts)
conn = duckdb.connect('../lakehouse_db/aki_lakehouse.db', read_only=True)
print('Connected to Lakehouse Database successfully.')

Connected to Lakehouse Database successfully.


### Test 1: Single Patient Chronological View
Pick a single patient stay and display their hourly progression. We should see exactly 1 hour increments in `hr_index`, forward-filled labs, and growing `hours_since_last_X` staleness metrics when no new data is charted.

In [2]:
# Find a stay that has AKI within 6 hours at some point to see the target shift in action
sample_stay = conn.execute('''
    SELECT stay_id 
    FROM gold_hourly_clinical 
    WHERE aki_within_6h = 1 
    LIMIT 1
''').fetchone()[0]

chronological_view = conn.execute(f'''
    SELECT 
        hr_index,
        hr_timestamp,
        creatinine,
        hours_since_last_cr,
        uo_hourly,
        uo_ml_kg_hr,
        aki_within_6h
    FROM gold_hourly_clinical
    WHERE stay_id = '{sample_stay}'
    ORDER BY hr_index ASC
    LIMIT 15
''').df()
display(chronological_view)

,hr_index,hr_timestamp,creatinine,hours_since_last_cr,uo_hourly,uo_ml_kg_hr,aki_within_6h
0,0,2150-03-03 21:00:00,1.4,48,250.0,3.022975,0
1,1,2150-03-03 22:00:00,1.4,48,0.0,0.000000,1
2,2,2150-03-03 23:00:00,1.4,48,65.0,0.785973,1
3,3,2150-03-04 00:00:00,1.2,0,45.0,0.544135,1
4,4,2150-03-04 01:00:00,1.2,1,35.0,0.423216,1
5,5,2150-03-04 02:00:00,1.2,2,30.0,0.362757,1
6,6,2150-03-04 03:00:00,1.2,3,30.0,0.362757,1


### Test 2: Target Leakage Verification
Since we updated the pipeline to strictly filter out rows `>=` the onset of AKI (`t.hr_index < MIN(hr_index) WHERE is_aki_now=1`), the `is_aki_now` column should **always** be 0 in this training table! If it is 1, the model is cheating.

In [3]:
leakage_check = conn.execute('''
    SELECT 
        MAX(is_aki_now) as max_aki_flag,
        SUM(is_aki_now) as total_aki_events_in_training_data
    FROM gold_hourly_clinical
''').df()
display(leakage_check)

,max_aki_flag,total_aki_events_in_training_data
0,0,0.0


### Test 3: Data Completeness (No NULLs)
We explicitly backfilled initial gaps with admission baselines and physiological normals. Therefore, our core ML features must have exactly 0 NULL values.

In [4]:
null_check = conn.execute('''
    SELECT 
        SUM(CASE WHEN creatinine IS NULL THEN 1 ELSE 0 END) as null_creatinine,
        SUM(CASE WHEN heart_rate IS NULL THEN 1 ELSE 0 END) as null_hr,
        SUM(CASE WHEN hours_since_last_cr IS NULL THEN 1 ELSE 0 END) as null_staleness_cr,
        SUM(CASE WHEN history_diabetes IS NULL THEN 1 ELSE 0 END) as null_diabetes
    FROM gold_hourly_clinical
''').df()
display(null_check)

,null_creatinine,null_hr,null_staleness_cr,null_diabetes
0,0.0,0.0,0.0,0.0


### Test 4: Age Clipping (HIPAA Compliance)
Verify that no patients have biological ages of 300, and that the maximum age is strictly clipped at 90.

In [5]:
age_check = conn.execute('''
    SELECT 
        MAX(age) as max_age_in_dataset,
        COUNT(DISTINCT patient_id) FILTER (WHERE age = 90) as unique_patients_clipped
    FROM gold_hourly_clinical
''').df()
display(age_check)


,max_age_in_dataset,unique_patients_clipped
0,90,2797


### Test 5: Target Distribution
Check the class imbalance for our 6-hour predictive horizon. This shows how many hourly snapshots precede an imminent AKI event vs how many are 'safe' hours.

In [ ]:
target_dist = conn.execute('''
    SELECT 
        aki_within_6h as imminent_aki_flag,
        COUNT(*) as number_of_hourly_samples,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
    FROM gold_hourly_clinical
    GROUP BY aki_within_6h
''').df()
display(target_dist)

,imminent_aki_flag,number_of_hourly_samples,percentage
0,0,2083777,90.42
1,1,220784,9.58


: 